In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math


In [2]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=512,
        temperature=0.0,
        model=deployment
    )

In [3]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/MultiArthsampled_train.json')
CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/CCoT_prompt_example.txt").read()

In [4]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/MultiArth/CoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return num
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'][0])  # Ground truth

        prompt_q = (
            CoT_prompt_examples +
            '\nQ: ' + q + " Think step by step. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions accurately."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Answer
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        # === Write to Correct/Incorrect Logs
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            print("wrong")
            bad_fd.write("❌ Incorrect or Invalid\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/205 [00:01<03:27,  1.02s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/205 [00:02<03:51,  1.14s/it]

Accuracy: 2 / 2 = 100.00%


  1%|▏         | 3/205 [00:03<04:17,  1.27s/it]

...

Accuracy: 194 / 203 = 95.57%


100%|█████████▉| 204/205 [03:56<00:00,  1.09it/s]

Accuracy: 195 / 204 = 95.59%


100%|██████████| 205/205 [03:57<00:00,  1.16s/it]

Accuracy: 196 / 205 = 95.61%


In [5]:
import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/MultiArth/standard.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Keep digits, decimal, minus
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        q = d['question']
        a = float(d['correct'][0])  # Ground truth

        prompt_q = (
            Standard_prompt_examples +
            '\nAnswer this question: ' + q + " Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Main Parallel Processing ===
results = []
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                global acc
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")
    fd.write(f"Final Accuracy: {acc} / {total} = {acc / total:.2%}\n") 

  0%|          | 1/205 [00:00<01:08,  2.98it/s]

Accuracy: 0 / 1 = 0.00%
Accuracy: 1 / 2 = 50.00%
Accuracy: 1 / 3 = 33.33%
Accuracy: 2 / 4 = 50.00%
Accuracy: 3 / 5 = 60.00%
Accuracy: 4 / 6 = 66.67%
Accuracy: 5 / 7 = 71.43%
Accuracy: 6 / 8 = 75.00%
Accuracy: 6 / 9 = 66.67%
Accuracy: 7 / 10 = 70.00%
Accuracy: 8 / 11 = 72.73%
Accuracy: 9 / 12 = 75.00%


  6%|▋         | 13/205 [00:00<00:07, 26.40it/s]

Accuracy: 10 / 13 = 76.92%
Accuracy: 11 / 14 = 78.57%
Accuracy: 12 / 15 = 80.00%
Accuracy: 12 / 16 = 75.00%
Accuracy: 13 / 17 = 76.47%
Accuracy: 13 / 18 = 72.22%
Accuracy: 14 / 19 = 73.68%
Accuracy: 15 / 20 = 75.00%
Accuracy: 16 / 21 = 76.19%


 19%|█▊        | 38/205 [00:01<00:03, 49.57it/s]

...

Accuracy: 125 / 153 = 81.70%
Accuracy: 126 / 154 = 81.82%
Accuracy: 127 / 155 = 81.94%
Accuracy: 128 / 156 = 82.05%
Accuracy: 129 / 157 = 82.17%
Accuracy: 130 / 158 = 82.28%
Accuracy: 131 / 159 = 82.39%
Accuracy: 131 / 160 = 81.88%
Accuracy: 132 / 161 = 81.99%
Accuracy: 132 / 162 = 81.48%
Accuracy: 133 / 163 = 81.60%
Accuracy: 134 / 164 = 81.71%
Accuracy: 135 / 165 = 81.82%
Accuracy: 136 / 166 = 81.93%
Accuracy: 136 / 167 = 81.44%
Accuracy: 136 / 168 = 80.95%
Accuracy: 137 / 169 = 81.07%
Accuracy: 137 / 170 = 80.59%
Accuracy: 138 / 171 = 80.70%
Accuracy: 139 / 172 = 80.81%
Accuracy: 140 / 173 = 80.92%
Accuracy: 141 / 174 = 81.03%
Accuracy: 142 / 175 = 81.14%
Accuracy: 143 / 176 = 81.25%
Accuracy: 143 / 177 = 80.79%
Accuracy: 144 / 178 = 80.90%
Accuracy: 145 / 179 = 81.01%
Accuracy: 146 / 180 = 81.11%
Accuracy: 146 / 181 = 80.66%
Accuracy: 146 / 182 = 80.22%
Accuracy: 147 / 183 = 80.33%
Accuracy: 148 / 184 = 80.43%


 94%|█████████▎| 192/205 [01:11<00:04,  2.85it/s]

Accuracy: 149 / 185 = 80.54%
Accuracy: 150 / 186 = 80.65%
Accuracy: 151 / 187 = 80.75%
Accuracy: 152 / 188 = 80.85%
Accuracy: 153 / 189 = 80.95%
Accuracy: 153 / 190 = 80.53%
Accuracy: 154 / 191 = 80.63%
Accuracy: 155 / 192 = 80.73%
Accuracy: 156 / 193 = 80.83%
Accuracy: 157 / 194 = 80.93%
Accuracy: 158 / 195 = 81.03%
Accuracy: 158 / 196 = 80.61%
Accuracy: 159 / 197 = 80.71%
Accuracy: 160 / 198 = 80.81%


100%|██████████| 205/205 [01:11<00:00,  2.85it/s]

Accuracy: 161 / 199 = 80.90%
Accuracy: 162 / 200 = 81.00%
Accuracy: 163 / 201 = 81.09%
Accuracy: 164 / 202 = 81.19%
Accuracy: 165 / 203 = 81.28%
Accuracy: 166 / 204 = 81.37%
Accuracy: 167 / 205 = 81.46%


In [8]:
import traceback
import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/MultiArth/complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
error_log_path = output_path.replace('.txt', '_errors.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(idx, d):
    global acc, total
    try:
        q = d['question']
        a = float(d['correct'][0])  # Ground truth answer

        # === Prompt Setup for Complex CoT ===
        prompt_q = (
            CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Please reason through this problem using a complex, multi-step chain of thought:\n"
            "Step 1: Clearly state all given information and any assumptions.\n"
            "Step 2: Propose two different methods to solve the problem, briefly outlining the logic of each.\n"
            "Step 3: For each method, work through all intermediate steps in detail, showing calculations, checks, and potential pitfalls.\n"
            "Step 4: Evaluate and compare the two methods—discussing which is better based on clarity, reliability, or efficiency.\n"
            "Step 5: Choose the better method and use it to solve the problem, showing all steps.\n"
            "Step 6: Double-check the solution for errors or unreasonable results.\n"
            "Finish your response with: the answer is <answer>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "Your goal is to answer the question using a complex, coherent, step by step thoughts, answering the questions correctly.\n"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Get Response ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ Incorrect or Invalid\n" + log_block

    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error at index {idx}:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
results = []
start_index = 129  # Start processing from question 127
with open(output_path, 'a') as fd, open(bad_output_path, 'a') as bad_fd, open(error_log_path, 'a') as error_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, idx, d) for idx, d in enumerate(dev_data[start_index:], start=start_index)]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                error_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

    # Final accuracy report
    fd.write(f"\nFinal Accuracy: {acc} / {total} = {acc / total:.2%}\n")
    fd.write(f"Final Accuracy: {acc} / {total} = {acc / total:.2%}\n")  # Write to the output file


  1%|▏         | 1/76 [00:02<03:44,  3.00s/it]

Accuracy: 1 / 1 = 100.00%


  3%|▎         | 2/76 [00:03<01:40,  1.36s/it]

Accuracy: 2 / 2 = 100.00%


  4%|▍         | 3/76 [00:04<01:20,  1.11s/it]

...

Accuracy: 57 / 63 = 90.48%
Accuracy: 58 / 64 = 90.62%
Accuracy: 59 / 65 = 90.77%
Accuracy: 60 / 66 = 90.91%
Accuracy: 60 / 67 = 89.55%


 89%|████████▉ | 68/76 [01:08<00:08,  1.02s/it]

Accuracy: 61 / 68 = 89.71%
Accuracy: 62 / 69 = 89.86%
Accuracy: 62 / 70 = 88.57%


100%|██████████| 76/76 [01:10<00:00,  1.08it/s]

Accuracy: 63 / 71 = 88.73%
Accuracy: 64 / 72 = 88.89%
Accuracy: 65 / 73 = 89.04%
Accuracy: 66 / 74 = 89.19%
Accuracy: 67 / 75 = 89.33%
Accuracy: 68 / 76 = 89.47%
